# An Analysis As A Description

**Version**: 1

**Last modified**: 2026-10-12

**Compatible with opengate-gate-tree version:** 0.9.0

**Author**: Mateusz Jakub Bała

## About

A question about events - the events in which two annihilation gammas were
recorded, and nothing else - is usually answered by a script that reads a
file, groups the rows and filters them. The question and the file are tangled
together: to ask it of another file, the script is run again; to ask a slightly
different one, it is copied and changed.

`EventTrackHitAnalysis` keeps them apart. An analysis is written once, as a
chain of operations, and kept as a description that reads nothing. It is
answered only when it is given data, and the same description answers on any
file. This notebook writes one, looks at it, reuses part of it in two others,
and answers all of them.

## What will you learn from this notebook

- write an analysis as a chain of operations, and read it back as it was
  written and in words
- see that building an analysis reads nothing, and that every call returns a
  new analysis
- reuse a part of an analysis in two others without either changing it
- answer an analysis with `apply`, which gives the events, tracks and hits it
  kept, and with `extract`, which gives a table

## Tutorial

Every step below is a line of text, the code that does what the line says, and
a word about the answer where the answer needs one.

In [1]:
from tools import environment, paths

print(environment.describe())

opengate-gate-tree: 0.8.0
python: 3.14.5
run on: 2026-10-12


The header of this notebook states the version it was written for; the line
above states the version it just ran against. If they differ, the notebook may
still be right — but nobody has checked it against what is installed.

The example file carried in the repository, described in
[`../data/README.md`](../data/README.md): 500 hits of a scene with a
`PositroniumSource` configured with every channel at once, beside a
back-to-back source. Nothing is written, so no output directory is needed.

In [2]:
from opengate_gate_tree import (
    EventTrackHit,
    EventTrackHitAnalysis,
    EventTrackHitField,
    GammaType,
    GateTree,
    Process,
    read_tree,
)

Analysis = EventTrackHitAnalysis
Field = EventTrackHitField
data = read_tree(paths.input_example("positronium-source-all-variants.root"), GateTree.HITS)
representation = EventTrackHit(data)
representation

EventTrackHit(events=115, tracks=311, hits=500)

### Writing An Analysis

An analysis starts at the events. `tracks(2)` keeps the events with two tracks
and moves to them, `gamma_type(...)` keeps the annihilation gammas among them,
and the closing `tracks(2)` keeps the events in which both of them were:

In [3]:
pairs = Analysis().tracks(2).gamma_type(GammaType.ANNIHILATION).tracks(2)
pairs

EventTrackHitAnalysis().tracks(2).gamma_type(GammaType.ANNIHILATION).tracks(2)

The analysis prints as the chain it was written as. `describe()` says the same
in words, one step per line:

In [4]:
print(pairs.describe())

1. Keep the events whose number of tracks passes eq=2, and move to their tracks.
2. Keep the tracks that pass gamma_type(GammaType.ANNIHILATION).
3. Keep the events whose number of tracks passes eq=2, and move to their tracks.


### Nothing Is Read Yet

`pairs` holds no data and has read none: it is a description. It is answered
when it is given some. `apply` returns the events, tracks and hits it kept, as
a representation of the same file:

In [5]:
kept = pairs.apply(representation)
kept

EventTrackHit(events=35, tracks=70, hits=124)

35 of the 115 events: the para-positronium decays without a prompt gamma and
the direct annihilations. The back-to-back events also left two gammas, but
not annihilation gammas of a positronium source, so their tracks were left out,
and the closing count left out the events with them.

The answer is an ordinary representation, so everything it offers works on it:

In [6]:
kept.tracks_frame([Field.SOURCE_TYPE, Field.TRACK_ID, Field.GAMMA_TYPE, Field.HIT_COUNT]).head()

,eventIndex,sourceType,trackID,gammaType,hitCount
trackIndex,,,,,
0,0,2,1,2,2
1,0,2,2,2,2
2,1,4,1,2,1
3,1,4,2,2,4
4,2,2,1,2,3


### Every Call Returns A New Analysis

An analysis never changes. Calling one of its methods gives a new analysis and
leaves the old one as it was, so a part shared by two questions is written
once:

In [7]:
scattered = pairs.hits(ge=2)
absorbed_at_once = pairs.hit(1).process(Process.PHOTOELECTRIC)

pairs, scattered, absorbed_at_once

(EventTrackHitAnalysis().tracks(2).gamma_type(GammaType.ANNIHILATION).tracks(2),
 EventTrackHitAnalysis().tracks(2).gamma_type(GammaType.ANNIHILATION).tracks(2).hits(ge=2),
 EventTrackHitAnalysis().tracks(2).gamma_type(GammaType.ANNIHILATION).tracks(2).hit(1).process(Process.PHOTOELECTRIC))

In [8]:
pairs.apply(representation), scattered.apply(representation), absorbed_at_once.apply(representation)

(EventTrackHit(events=35, tracks=70, hits=124),
 EventTrackHit(events=29, tracks=35, hits=89),
 EventTrackHit(events=29, tracks=35, hits=35))

`pairs` still keeps its 35 events. `scattered` keeps the annihilation gammas
of those events that interacted at least twice, and `absorbed_at_once` the ones
whose first interaction was a photoelectric absorption - in 29 events each, and
35 gammas each: the two questions split the 70 gammas between them.

### A Description Answers On Any Data

The same analysis takes a representation, the columns a reader returns, or a
frame of them; the representation is built inside when it is not given:

In [9]:
from_columns = pairs.apply(data)
from_frame = pairs.apply(data.to_dataframe())

from_columns.hit_count == from_frame.hit_count == kept.hit_count

True

Two analyses written the same way are equal, and can key a dictionary of
answers; a count given as `tracks(2)` or as `tracks(eq=2)` asks the same:

In [10]:
(
    Analysis().tracks(2) == Analysis().tracks(eq=2),
    pairs == Analysis().tracks(2).gamma_type(GammaType.ANNIHILATION).tracks(2),
)

(True, True)

### A Table Instead Of A Representation

`extract` performs the analysis and returns a table with one row per event,
the variables asked for in its columns. The prompt gammas, the source model of
their event, and the energy of their first interaction:

In [11]:
prompt = Analysis().tracks().gamma_type(GammaType.PROMPT)
prompt.extract(representation, [Field.SOURCE_TYPE, Field.TRACK_ID, Field.EDEP]).head()

,runID,eventID,sourceType,t1_trackID,t1h1_edep
0,0,4,3,1,0.712212
1,0,6,3,1,0.557122
2,0,11,2,1,1.300000
3,0,13,3,1,1.300000
4,0,18,2,1,1.300000


A prompt gamma is always track 1, the first particle the source emitted. The
table is the subject of a tutorial of its own; it is enough here that it comes
from the same description.

## Summary

An analysis is a chain of operations kept as a description: it prints as it
was written, says its steps in words, reads nothing until `apply` or `extract`
gives it data, and never changes, so one part of it can be reused in several
questions and every question asked of any file.

## Bibliography

- [Analysis Of Events, Tracks And Hits](https://opengate-gate-tree.readthedocs.io/en/latest/guide/analysis.html)
  — an analysis as a description, `apply` and `extract`
- [Events, Tracks And Hits](https://opengate-gate-tree.readthedocs.io/en/latest/guide/event-track-hit.html)
  — the representation an analysis is performed on, and the order it counts in
- [PositroniumSource Data](https://opengate-gate-tree.readthedocs.io/en/latest/guide/positronium.html)
  — what `sourceType`, `decayType` and `gammaType` mean

## Changelog

- v1: written.